In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'resume-dataset' dataset.
Path to dataset files: /kaggle/input/resume-dataset


In [2]:
import os
print(os.listdir(path))

['Resume', 'data']


In [5]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [10]:
n = os.path.join(path, "Resume")
print(n)
new_path=os.path.join(n,"Resume.csv")
print(new_path)

/kaggle/input/resume-dataset/Resume
/kaggle/input/resume-dataset/Resume/Resume.csv


In [20]:
import pandas as pd

df = pd.read_csv(new_path)

df = df[['Resume_str', 'Category']]

print(df.shape)
df.head()

(2484, 2)


,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [21]:
print(df['Category'].value_counts())

print(df['Category'].nunique())

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64
24


In [22]:
import re
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", " ", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)

df["clean_resume"] = df["Resume_str"].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["Category"])

print(encoder.classes_)

['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_resume"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [25]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 10000

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [26]:
MAX_LEN = 300

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post'
)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

num_classes = len(df["label"].unique())

model = Sequential()

model.add(
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LEN
    )
)

model.add(
    LSTM(
        128,
        dropout=0.2,
        recurrent_dropout=0.2
    )
)

model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))

model.add(Dense(num_classes, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.0522 - loss: 3.1642 - val_accuracy: 0.0427 - val_loss: 3.1633
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 74s 845ms/step - accuracy: 0.1001 - loss: 3.0583 - val_accuracy: 0.0829 - val_loss: 3.0598
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 44s 876ms/step - accuracy: 0.1403 - loss: 2.8649 - val_accuracy: 0.1156 - val_loss: 3.0056
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 44s 873ms/step - accuracy: 0.1756 - loss: 2.6661 - val_accuracy: 0.1508 - val_loss: 2.9832
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 44s 874ms/step - accuracy: 0.2329 - loss: 2.4199 - val_accuracy: 0.1432 - val_loss: 3.1690
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 43s 872ms/step - accuracy: 0.3121 - loss: 2.2039 - val_accuracy: 0.1533 - val_loss: 3.2072
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 44s 876ms/step - accuracy: 0.3820 - loss: 1.9217 - val_accuracy: 0.1508 - val_loss: 3.3576
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 82s 872ms/step - accuracy: 0.4563 - loss: 1.7051 - val_accurac

In [30]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Accuracy:", accuracy)

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 104ms/step - accuracy: 0.1610 - loss: 3.7512
Accuracy: 0.16096580028533936


In [31]:
model.save("resume_lstm.h5")


In [32]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

In [33]:
import numpy as np

sample = """
Python
Machine Learning
Deep Learning
TensorFlow
NLP
Pandas
NumPy
"""

sample = clean_text(sample)

seq = tokenizer.texts_to_sequences([sample])

pad = pad_sequences(
    seq,
    maxlen=MAX_LEN,
    padding='post'
)

pred = model.predict(pad)

label = np.argmax(pred)

print(
    encoder.inverse_transform([label])[0]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step
SALES


In [34]:
pred = model.predict(pad)[0]

top3 = np.argsort(pred)[::-1][:3]

for i in top3:
    print(
        encoder.inverse_transform([i])[0],
        round(pred[i]*100,2),
        "%"
    )

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
SALES 15.61 %
DESIGNER 12.05 %
FITNESS 11.49 %
